# PlagIA Enterprise · 04 — Model Registry, Exportación ONNX (Servidor + Flutter INT8) y Mapas de Calor

**Subsistema de Despliegue MLOps, Edge-AI y Análisis Geoespacial (`ml_engine`)**

### Objetivos Técnicos
1. Consultar y comparar los experimentos en **MLflow** priorizando **F1-Macro y Recall Fitosanitario**.
2. Registrar el modelo ganador en el **MLflow Model Registry** y promoverlo a `Staging`.
3. Exportar el modelo a **ONNX FP32 (`models/plagia_server_fp32.onnx`)** y cuantizarlo a **INT8 (`models/plagia_mobile_int8.onnx`)**, sincronizando automáticamente los assets hacia `mobile_app/assets/models/` para **Android Studio / Flutter**.
4. Ejecutar el benchmark de latencia **PyTorch vs. ONNX Runtime** y generar el **Mapa de Calor Fitosanitario GeoJSON y PNG** con **GeoPandas**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
from mlflow.tracking import MlflowClient

from ml_engine.architectures import build_pytorch_classifier
from ml_engine.config import (
    DRONE_CLASSES,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_MODEL_NAME,
    MLFLOW_TRACKING_URI,
)
from ml_engine.exporter import (
    benchmark_pytorch_vs_onnx,
    ensure_default_artifacts_exist,
)
from ml_engine.geospatial import (
    build_heatmap_geodataframe,
    export_heatmap_artifacts,
)

## 1. Selección del Mejor Run y Promoción en MLflow Model Registry

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

runs_df = mlflow.search_runs(order_by=["metrics.f1_macro DESC"])
if not runs_df.empty:
    best_run = runs_df.iloc[0]
    print("Mejor corrida seleccionada:", best_run.get("tags.mlflow.runName"))
    print("F1-Macro :", best_run.get("metrics.f1_macro"))
    print("Recall   :", best_run.get("metrics.recall_macro"))
    model_uri = f"runs:/{best_run['run_id']}/model"
    reg = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
    client = MlflowClient()
    client.set_registered_model_alias(MLFLOW_MODEL_NAME, "staging", reg.version)
    print(f"Modelo '{MLFLOW_MODEL_NAME}' v{reg.version} promovido al alias '@staging'.")
else:
    print("No hay runs previos en MLflow; se exportará el artefacto base para validación.")

## 2. Exportación a ONNX (Servidor + Mobile INT8 para Flutter) y Benchmark

In [ ]:
export_meta = ensure_default_artifacts_exist(class_names=DRONE_CLASSES)
print("Metadatos de exportación ONNX:")
print(" - Servidor ONNX (MB):", export_meta["server_size_mb"])
print(" - Mobile INT8 (MB)  :", export_meta["mobile_size_mb"])

ref_model = build_pytorch_classifier("efficientnet_b0", num_classes=len(DRONE_CLASSES), pretrained=False)
bench = benchmark_pytorch_vs_onnx(ref_model)
print("\nBenchmark de Inferencia:")
for k, v in bench.items():
    print(f" - {k}: {v}")

## 3. Generación del Mapa de Calor Fitosanitario con GeoPandas (GeoJSON + PNG)

In [ ]:
sample_detections = [
    {"patch_id": "P01", "disease_label": "sano", "confidence": 0.96, "bbox_gps": (-74.895, 4.142, -74.890, 4.147)},
    {"patch_id": "P02", "disease_label": "roya_parda", "confidence": 0.91, "bbox_gps": (-74.890, 4.142, -74.885, 4.147)},
    {"patch_id": "P03", "disease_label": "roya_amarilla", "confidence": 0.89, "bbox_gps": (-74.885, 4.142, -74.880, 4.147)},
    {"patch_id": "P04", "disease_label": "sano", "confidence": 0.94, "bbox_gps": (-74.895, 4.147, -74.890, 4.152)},
    {"patch_id": "P05", "disease_label": "roya_parda", "confidence": 0.95, "bbox_gps": (-74.890, 4.147, -74.885, 4.152)},
    {"patch_id": "P06", "disease_label": "sano", "confidence": 0.92, "bbox_gps": (-74.885, 4.147, -74.880, 4.152)},
]

gdf = build_heatmap_geodataframe(sample_detections, lot_id="LOTE-ESPINAL-01", municipio="El Espinal")
heatmap_info = export_heatmap_artifacts(gdf, lot_id="LOTE-ESPINAL-01")
print("Mapa de calor exportado:")
print(" - GeoJSON :", heatmap_info["geojson_path"])
print(" - PNG     :", heatmap_info["png_path"])
print(" - Tasa de infestación del lote:", f"{heatmap_info['infection_rate']*100:.1f}%")